# Phase 1 sanity check

End-to-end plumbing: text -> Model A hidden state -> random orthogonal adapter -> Model B `inputs_embeds` -> generated text.

Output text will be garbage. That is correct: the adapter is untrained. What we are verifying:

1. Both models load on the detected device.
2. Hidden-state extraction returns the expected shape.
3. The adapter projects A's space to B's space.
4. `inputs_embeds` injection generates without error.
5. Backprop through the chain (A frozen -> adapter -> B frozen -> loss) produces a finite, nonzero gradient on the adapter.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import torch

from src.adapter import Adapter, init_orthogonal
from src.models import (
    DEFAULT_MODEL_A,
    DEFAULT_MODEL_B,
    decode_from_hidden,
    encode_to_hidden,
    load_frozen,
    pick_device,
)

device = pick_device()
device

In [ ]:
model_a, tok_a = load_frozen(DEFAULT_MODEL_A)
if DEFAULT_MODEL_B == DEFAULT_MODEL_A:
    model_b, tok_b = model_a, tok_a
else:
    model_b, tok_b = load_frozen(DEFAULT_MODEL_B)

d_a = model_a.config.hidden_size
d_b = model_b.config.hidden_size
print(f"d_A = {d_a},  d_B = {d_b}")

In [ ]:
hidden_a = encode_to_hidden(model_a, tok_a, "Hello world.")
print("hidden_a:", tuple(hidden_a.shape), hidden_a.dtype)

In [ ]:
adapter = Adapter(d_a, d_b).to(device).to(hidden_a.dtype)
init_orthogonal(adapter)

with torch.no_grad():
    hidden_b = adapter(hidden_a)
print("hidden_b:", tuple(hidden_b.shape), hidden_b.dtype)

In [ ]:
text, debug = decode_from_hidden(model_b, tok_b, hidden_b, max_new_tokens=30)
print("debug:", debug)
print("generated:", repr(text))

In [ ]:
# Gradient sanity: enable grads on the adapter, push through B, backprop.
for p in adapter.parameters():
    p.requires_grad_(True)

hidden_a_ng = encode_to_hidden(model_a, tok_a, "Gradient probe sentence.")
hidden_b_grad = adapter(hidden_a_ng)

from src.models import _split_chat_template

pre, post = _split_chat_template(tok_b, "You received a signal:\n")
pre_ids = tok_b(pre, return_tensors="pt", add_special_tokens=False).input_ids.to(model_b.device)
post_ids = tok_b(post, return_tensors="pt", add_special_tokens=False).input_ids.to(model_b.device)
embed = model_b.get_input_embeddings()
pre_embeds = embed(pre_ids)
post_embeds = embed(post_ids)
vec_embed = hidden_b_grad.to(pre_embeds.dtype).reshape(1, 1, -1)
inputs_embeds = torch.cat([pre_embeds, vec_embed, post_embeds], dim=1)
attn = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=model_b.device)

out = model_b(inputs_embeds=inputs_embeds, attention_mask=attn)
loss = out.logits[:, -1, :].float().log_softmax(dim=-1)[:, 0].mean()
loss.backward()

g = adapter.linear.weight.grad
print("grad shape:", tuple(g.shape))
print("finite:", torch.isfinite(g).all().item())
print("nonzero:", (g.abs().sum() > 0).item())
print("grad norm:", g.float().norm().item())